# Stage 04b: Train/Test Split

**Purpose:** Split data by fold

**Outputs:** data/04b_train.parquet, data/04b_test.parquet

In [ ]:
config_path = "config/car_coll/v1"

In [ ]:
import pandas as pd
import yaml
import os
import sys
import gc
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / 'lib'))
from utils import setup_notebook_environment

print("Stage 04b: Train/test split")

project_root = setup_notebook_environment()

In [ ]:
config_file = f"{config_path}/config.yaml"
with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)

pc_num = open('current.pc').read().strip()
pc_id = f'PC{pc_num}'
output_base = cfg['machines'][pc_id]['paths']['output_path']

In [ ]:
checkpoint_02 = f"{output_base}/data/02_conditioned.parquet"
data = pd.read_parquet(checkpoint_02)
print(f"\nLoaded: {data.shape}")

# Merge PCA features if enabled
pca_cfg = cfg.get('feature_engineering', {}).get('pca', {})
pca_enabled = pca_cfg.get('enabled', False)

if pca_enabled:
    pca_file = f"{output_base}/data/04a_pca_features.parquet"
    if os.path.exists(pca_file):
        pca_features = pd.read_parquet(pca_file)
        join_key = cfg['data']['join_key']
        data = data.merge(pca_features, on=join_key, how='left')
        n_pca_cols = len(pca_features.columns) - 1
        print(f"  Merged {n_pca_cols} PCA features")

# Validate critical columns exist
critical_cols = [
    cfg['data']['join_key'],
    cfg['data']['fold_column'],
    cfg['experiment']['target'],
    cfg['experiment']['exposure']
]
missing = [c for c in critical_cols if c not in data.columns]
if missing:
    raise ValueError(f"Missing critical columns: {missing}")

print(f"\nFinal shape: {data.shape}")
print(f"Critical columns validated: {', '.join(critical_cols)}")

In [ ]:
# Split by fold
mode = cfg['run_mode']
fold_col = cfg['data']['fold_column']
test_folds = cfg[mode]['test_folds']
train_folds = cfg[mode]['train_folds']

train_data = data[data[fold_col].isin(train_folds)].copy()
test_data = data[data[fold_col].isin(test_folds)].copy()

print(f"\nTrain: {train_data.shape}")
print(f"Test: {test_data.shape}")

del data
gc.collect()

In [ ]:
from utils import validate_critical_columns
validate_critical_columns(train_data, cfg, 'Stage 04 Train Output')
validate_critical_columns(test_data, cfg, 'Stage 04 Test Output')

# vin_date as index so later stages align on it
join_key = cfg['data']['join_key']
train_data = train_data.set_index(join_key)
test_data = test_data.set_index(join_key)
print(f"\nSet {join_key} as index")

# Save splits
train_file = f"{output_base}/data/04b_train.parquet"
test_file = f"{output_base}/data/04b_test.parquet"

from utils import save_parquet
save_parquet(train_data, train_file)
save_parquet(test_data, test_file)

print(f"\nSaved: {train_file}")
print(f"Saved: {test_file}")

In [ ]:
print("Stage 04b done")